# Apache Hudi - Incremental Queries & CDC

Apache Hudi supports incremental queries, which allow readers to process only records changed between commit instants.
This pattern is useful for Change Data Capture (CDC), downstream synchronization, streaming-style processing, and efficient ETL pipelines that should not scan the full table every time.

## What you will learn

In this notebook, you will learn:

- Creating a Hudi Copy-On-Write (COW) table for CDC-style data
- Writing multiple commits to simulate changes over time
- Capturing Hudi commit instants from metadata columns
- Reading only changed records using incremental query mode
- Using begin and end commit boundaries
- Understanding low isolation level reads
- Building a simple CDC-style output from Hudi metadata

## Step 1 — Create Spark session

This notebook assumes that the Hudi bundle JAR is already available in the Spark Docker image under `/opt/spark/jars`.

Because of that, we do not use `spark.jars.packages` here. This avoids Ivy/Maven downloads from inside the notebook.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Hudi-07-Incremental-Queries-CDC")
    .master("spark://spark-master:7077")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.2


## Step 2 — Define shared paths and table name

This notebook is independent and creates its own table. It does not require any table from previous notebooks.

In [2]:
BASE_PATH = "/workspace/data/hudi"
TABLE_NAME = "riders_incremental_cdc"
TABLE_PATH = f"{BASE_PATH}/{TABLE_NAME}"

print("Table name:", TABLE_NAME)
print("Table path:", TABLE_PATH)

Table name: riders_incremental_cdc
Table path: /workspace/data/hudi/riders_incremental_cdc


## Step 3 — Clean previous run

For a tutorial notebook, we remove the previous table path so that reruns start from a known state.

In [3]:
import shutil
import os

if os.path.exists(TABLE_PATH):
    shutil.rmtree(TABLE_PATH)
    print(f"Removed existing table path: {TABLE_PATH}")
else:
    print("No existing table path found.")

No existing table path found.


## Step 4 — Define Hudi write options

The table uses:

- `rider` as the record key
- `city` as the partition path
- `ts` as the precombine field
- `COPY_ON_WRITE` table type

In [4]:
hudi_write_options = {
    "hoodie.table.name": TABLE_NAME,
    "hoodie.datasource.write.table.name": TABLE_NAME,
    "hoodie.datasource.write.recordkey.field": "rider",
    "hoodie.datasource.write.partitionpath.field": "city",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.operation": "upsert",
}

## Step 5 — Write the initial dataset

This creates the first commit. In CDC terms, this is the baseline load.

In [5]:
from pyspark.sql.functions import to_timestamp

df_initial = spark.createDataFrame(
    [
        ("r6", "BOS", "2024-01-08 09:00:00"),
        ("r7", "CHI", "2024-01-08 09:10:00"),
    ],
    ["rider", "city", "ts"]
).withColumn("ts", to_timestamp("ts"))

df_initial.show(truncate=False)

(
    df_initial.write
    .format("hudi")
    .options(**hudi_write_options)
    .mode("overwrite")
    .save(TABLE_PATH)
)

print("Initial write completed.")

+-----+----+-------------------+
|rider|city|ts                 |
+-----+----+-------------------+
|r6   |BOS |2024-01-08 09:00:00|
|r7   |CHI |2024-01-08 09:10:00|
+-----+----+-------------------+



# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
# WARNING: Unable to attach Serviceability Agent. You can try again with escalated privileges. Two options: a) use -Djol.tryWithSudo=true to try with sudo; b) echo 0 | sudo tee /proc/sys/kernel/yama/ptrace_scope


[Stage 35:>                                                         (0 + 2) / 2]

26/04/25 19:50:07 WARN HoodieTableFileSystemView: Partition: CHI is not available in store
26/04/25 19:50:07 WARN HoodieTableFileSystemView: Partition: BOS is not available in store


Initial write completed.


## Step 6 — Capture the baseline commit instant

We read `_hoodie_commit_time` from the table metadata. This commit instant can be used as a checkpoint for incremental reads.

In [6]:
baseline_df = spark.read.format("hudi").load(TABLE_PATH)

baseline_commits = [
    row["_hoodie_commit_time"]
    for row in baseline_df
        .select("_hoodie_commit_time")
        .distinct()
        .orderBy("_hoodie_commit_time")
        .collect()
]

if not baseline_commits:
    raise RuntimeError("No baseline Hudi commit found.")

baseline_commit = baseline_commits[-1]

print("Baseline commit instant:", baseline_commit)

baseline_df.select(
    "_hoodie_commit_time",
    "_hoodie_record_key",
    "rider",
    "city",
    "ts"
).orderBy("rider").show(truncate=False)

Baseline commit instant: 20260425194949476
+-------------------+------------------+-----+----+-------------------+
|_hoodie_commit_time|_hoodie_record_key|rider|city|ts                 |
+-------------------+------------------+-----+----+-------------------+
|20260425194949476  |r6                |r6   |BOS |2024-01-08 09:00:00|
|20260425194949476  |r7                |r7   |CHI |2024-01-08 09:10:00|
+-------------------+------------------+-----+----+-------------------+



## Step 7 — Write CDC-style changes

This simulates a CDC batch:

- `r6` is updated with a newer timestamp
- `r8` is inserted as a new rider

In [7]:
import time

time.sleep(1)

df_changes = spark.createDataFrame(
    [
        ("r6", "BOS", "2024-01-08 10:00:00"),
        ("r8", "MIA", "2024-01-08 10:05:00"),
    ],
    ["rider", "city", "ts"]
).withColumn("ts", to_timestamp("ts"))

df_changes.show(truncate=False)

(
    df_changes.write
    .format("hudi")
    .options(**hudi_write_options)
    .mode("append")
    .save(TABLE_PATH)
)

print("CDC-style change batch written.")

+-----+----+-------------------+
|rider|city|ts                 |
+-----+----+-------------------+
|r6   |BOS |2024-01-08 10:00:00|
|r8   |MIA |2024-01-08 10:05:00|
+-----+----+-------------------+



[Stage 57:>                                                         (0 + 2) / 2]

26/04/25 19:50:22 WARN HoodieTableFileSystemView: Partition: MIA is not available in store


26/04/25 19:50:22 WARN HoodieTableFileSystemView: Partition: MIA is not available in store
26/04/25 19:50:25 WARN HoodieTableFileSystemView: Partition: MIA is not available in store
CDC-style change batch written.


## Step 8 — Read the latest snapshot

A normal Hudi read returns the current state of the table.

In [8]:
latest_df = spark.read.format("hudi").load(TABLE_PATH)
latest_df.createOrReplaceTempView("riders_cdc_latest")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time
FROM riders_cdc_latest
ORDER BY rider
""").show(truncate=False)

+-----+----+-------------------+-------------------+
|rider|city|ts                 |_hoodie_commit_time|
+-----+----+-------------------+-------------------+
|r6   |BOS |2024-01-08 10:00:00|20260425195017452  |
|r7   |CHI |2024-01-08 09:10:00|20260425194949476  |
|r8   |MIA |2024-01-08 10:05:00|20260425195017452  |
+-----+----+-------------------+-------------------+



## Step 9 — Capture all visible commit instants

The latest snapshot only contains current records, but the metadata still tells us which commit produced each current version.

In [9]:
commits = [
    row["_hoodie_commit_time"]
    for row in latest_df
        .select("_hoodie_commit_time")
        .distinct()
        .orderBy("_hoodie_commit_time")
        .collect()
]

if len(commits) < 2:
    raise RuntimeError("Expected at least two commits. Check that both write steps completed.")

print("Visible commit instants:")
for commit in commits:
    print(commit)

latest_commit = commits[-1]

print(f"\nBaseline checkpoint: {baseline_commit}")
print(f"Latest commit: {latest_commit}")

Visible commit instants:
20260425194949476
20260425195017452

Baseline checkpoint: 20260425194949476
Latest commit: 20260425195017452


## Step 10 — Incremental query after the baseline checkpoint

Incremental query mode reads records changed after a given begin instant.

This is the core CDC pattern: store the last processed commit instant, then read only newer changes.

In [10]:
cdc_incremental_df = (
    spark.read
    .format("hudi")
    .option("hoodie.datasource.query.type", "incremental")
    .option("hoodie.datasource.read.begin.instanttime", baseline_commit)
    .option("hoodie.datasource.read.end.instanttime", latest_commit)
    .option("hoodie.isolation.level", "low")
    .load(TABLE_PATH)
)

cdc_incremental_df.createOrReplaceTempView("riders_cdc_inc")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time,
    _hoodie_record_key,
    _hoodie_partition_path
FROM riders_cdc_inc
ORDER BY _hoodie_commit_time, rider
""").show(truncate=False)

26/04/25 19:50:35 WARN CacheManager: Asked to cache already cached data.
+-----+----+-------------------+-------------------+------------------+----------------------+
|rider|city|ts                 |_hoodie_commit_time|_hoodie_record_key|_hoodie_partition_path|
+-----+----+-------------------+-------------------+------------------+----------------------+
|r6   |BOS |2024-01-08 09:00:00|20260425194949476  |r6                |BOS                   |
|r7   |CHI |2024-01-08 09:10:00|20260425194949476  |r7                |CHI                   |
+-----+----+-------------------+-------------------+------------------+----------------------+



## Step 11 — Build a CDC-style output

This view keeps only the business columns and the Hudi commit instant.

In production, the commit instant is commonly stored as a checkpoint after successful downstream processing.

In [11]:
spark.sql("""
SELECT
    _hoodie_commit_time AS change_commit,
    rider,
    city,
    ts AS business_timestamp
FROM riders_cdc_inc
ORDER BY change_commit, rider
""").show(truncate=False)

+-----------------+-----+----+-------------------+
|change_commit    |rider|city|business_timestamp |
+-----------------+-----+----+-------------------+
|20260425194949476|r6   |BOS |2024-01-08 09:00:00|
|20260425194949476|r7   |CHI |2024-01-08 09:10:00|
+-----------------+-----+----+-------------------+



## Step 12 — Read the last N commits

Some pipelines use commit-count based reads. The option below asks Hudi to read the last delta commit window.

This is useful for demos, but production CDC usually prefers explicit commit checkpoints.

In [12]:
last_delta_df = (
    spark.read
    .format("hudi")
    .option("hoodie.datasource.read.delta.commits", "1")
    .option("hoodie.isolation.level", "low")
    .load(TABLE_PATH)
)

last_delta_df.createOrReplaceTempView("riders_cdc_last_delta")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time
FROM riders_cdc_last_delta
ORDER BY rider
""").show(truncate=False)

+-----+----+-------------------+-------------------+
|rider|city|ts                 |_hoodie_commit_time|
+-----+----+-------------------+-------------------+
|r6   |BOS |2024-01-08 10:00:00|20260425195017452  |
|r7   |CHI |2024-01-08 09:10:00|20260425194949476  |
|r8   |MIA |2024-01-08 10:05:00|20260425195017452  |
+-----+----+-------------------+-------------------+



## Step 13 — Inspect table files

This step lists files under the table path so you can see partition folders and Hudi metadata files.

In [13]:
from pathlib import Path

table_path = Path(TABLE_PATH)

print("Table files:")
for path in sorted(table_path.rglob("*")):
    if path.is_file():
        print(path.relative_to(table_path))

Table files:
.hoodie/.hoodie.properties.crc
.hoodie/.index_defs/.index.json.crc
.hoodie/.index_defs/index.json
.hoodie/hoodie.properties
.hoodie/metadata/.hoodie/.hoodie.properties.crc
.hoodie/metadata/.hoodie/hoodie.properties
.hoodie/metadata/.hoodie/timeline/.00000000000000000.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000000.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000000_20260425194953098.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001_20260425194955359.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002_20260425194959352.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.20260

## Step 14 — Summary

In this notebook, you created an independent Hudi table and used incremental queries to simulate a CDC pipeline.

Key takeaways:

- Incremental queries read only changed records between commit instants.
- `_hoodie_commit_time` can be used as a CDC checkpoint.
- Snapshot reads show the latest table state.
- Low isolation level can be useful for less strict incremental reads.
- Explicit begin/end commit boundaries are more reliable than relying only on a commit count.